In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
data = pd.read_csv("manufacturing.csv")

In [3]:
data.head()

,Temperature (°C),Pressure (kPa),Temperature x Pressure,Material Fusion Metric,Material Transformation Metric,Quality Rating
0,209.762701,8.050855,1688.769167,44522.217074,9.229576e+06,99.999971
1,243.037873,15.812068,3842.931469,63020.764997,1.435537e+07,99.985703
2,220.552675,7.843130,1729.823314,49125.950249,1.072839e+07,99.999758
3,208.976637,23.786089,4970.736918,57128.881547,9.125702e+06,99.999975
4,184.730960,15.797812,2918.345014,38068.201283,6.303792e+06,100.000000


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3957 entries, 0 to 3956
Data columns (total 6 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Temperature (°C)                3957 non-null   float64
 1   Pressure (kPa)                  3957 non-null   float64
 2   Temperature x Pressure          3957 non-null   float64
 3   Material Fusion Metric          3957 non-null   float64
 4   Material Transformation Metric  3957 non-null   float64
 5   Quality Rating                  3957 non-null   float64
dtypes: float64(6)
memory usage: 185.6 KB


In [5]:
data.corr()

,Temperature (°C),Pressure (kPa),Temperature x Pressure,Material Fusion Metric,Material Transformation Metric,Quality Rating
Temperature (°C),1.000000,-0.024754,0.571743,0.974901,0.971210,-0.461279
Pressure (kPa),-0.024754,1.000000,0.773572,0.151095,-0.022862,0.013129
Temperature x Pressure,0.571743,0.773572,1.000000,0.694733,0.555579,-0.258474
Material Fusion Metric,0.974901,0.151095,0.694733,1.000000,0.976708,-0.511972
Material Transformation Metric,0.971210,-0.022862,0.555579,0.976708,1.000000,-0.575756
Quality Rating,-0.461279,0.013129,-0.258474,-0.511972,-0.575756,1.000000


In [7]:
data = data.drop("Temperature x Pressure",axis=1)

In [8]:
data.head()

,Temperature (°C),Pressure (kPa),Material Fusion Metric,Material Transformation Metric,Quality Rating
0,209.762701,8.050855,44522.217074,9.229576e+06,99.999971
1,243.037873,15.812068,63020.764997,1.435537e+07,99.985703
2,220.552675,7.843130,49125.950249,1.072839e+07,99.999758
3,208.976637,23.786089,57128.881547,9.125702e+06,99.999975
4,184.730960,15.797812,38068.201283,6.303792e+06,100.000000


In [9]:
# Temperature x Pressure çıkartma sebebim: Halihazırda Temperature ve Pressure kolonları ile yüksek korelasyona sahip olmasıdır.
# Çıkartılmasaydı eğer overfitting'e sebep olabilirdi.

In [10]:
# Temizlenmesi veya düzeltilmesi gereken başka bir şey yok. Sırasıyla Logistic, SVR, KNN, Decision Tree regresyonlarını kıyas yapacağım.

In [11]:
## Scale İşlemi

In [12]:
from sklearn.preprocessing import StandardScaler

In [14]:
scaler = StandardScaler()

In [15]:
from sklearn.model_selection import train_test_split

In [16]:
X = data.drop("Quality Rating",axis=1)
y = data["Quality Rating"]

In [17]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=15)

In [18]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [19]:
from sklearn.linear_model import LinearRegression

In [20]:
reg = LinearRegression()

In [22]:
from sklearn.metrics import r2_score

In [24]:
reg.fit(X_train_scaled,y_train)
y_pred = reg.predict(X_test_scaled)
print("Score:",r2_score(y_test,y_pred))

Score: 0.49897085592909785


In [41]:
from sklearn.linear_model import Ridge

In [42]:
param = {
    "alpha" : [0.01,0.1,1.0,10.0,100.0],
    "solver" : ['auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs']
}

In [44]:
from sklearn.model_selection import GridSearchCV

In [45]:
grid = GridSearchCV(estimator=Ridge(),param_grid=param, cv=5)

In [77]:
import warnings
warnings.filterwarnings('ignore')
grid.fit(X_train_scaled,y_train)
y_pred = grid.predict(X_test_scaled)
print("Score:",r2_score(y_test,y_pred))

Score: 0.4982786267376689


In [68]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

In [69]:
# Pipeline

In [72]:

def poly_regression(degree):
    poly_features = PolynomialFeatures(degree=degree)
    lin_reg = LinearRegression()
    scaler = StandardScaler()
    pipeline = Pipeline([
        ("standard_scaler", scaler),
        ("poly_features", poly_features),
        ("lin_reg", lin_reg)
    ])
    pipeline.fit(X_train, y_train)
    score = pipeline.score(X_test, y_test)
    print("R2 score: ", score)

In [73]:
for degree in [1,2,3,4,5]:
    poly_regression(degree)

R2 score:  0.49897085592909785
R2 score:  0.9292154638502551
R2 score:  0.9969607071152742
R2 score:  0.9999637324440812
R2 score:  0.9999998840054525


In [75]:
# Verimiz doğrusal yapıda olmadığıdan dolayı geçerli skoru elde edememiştik. PolynomialFeature sayesinde Pipeline ile daha iyi sonuçlar elde ettik

In [78]:
#SVR

In [80]:
from sklearn.svm import SVR

In [81]:
svr = SVR(kernel="linear") #Bilerek linear sectim. Sonuc düsük gelmesi gerekiyor.

In [83]:
# Önceki verilerle karismamasi icin tekrar scale ediyorum

In [84]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [86]:
svr.fit(X_train_scaled,y_train)
y_pred = svr.predict(X_test_scaled)
print("Score:",r2_score(y_test,y_pred))

Score: 0.07398707739003518


In [87]:
# Beklediğimiz gibi çok düşük geldi. Şimdi hyperparameter tuning işlemi yapalım.

In [ ]:
param = {
    'kernel' : ['poly', 'rbf', 'sigmoid'],
    'gamma' : ["auto","scale"],
    'C' : [0,0.1,1,10,100,1000],
}

In [ ]:
grid = GridSearchCV(estimator=SVR(),param_grid=param,cv=5,n_jobs=-1)

In [94]:
grid.best_params_

{'C': 1000, 'gamma': 'scale', 'kernel': 'rbf'}

In [95]:
y_pred = grid.predict(X_test_scaled)
print("Score:",r2_score(y_pred,y_test))

Score: 0.9965754023715911


In [96]:
# SVR ile yaptığımız grid işleminde en iyi kernel: rbf olarak seçilmiş.

In [97]:
# KNN

In [98]:
from sklearn.neighbors import KNeighborsRegressor

In [101]:
reg = KNeighborsRegressor()

In [102]:
reg.fit(X_train_scaled,y_train)
y_pred = reg.predict(X_test_scaled)

In [103]:
print("score:",r2_score(y_pred,y_test))

score: 0.9850915688995806


In [104]:
reg = KNeighborsRegressor(n_neighbors=7,algorithm='auto')
reg.fit(X_train_scaled,y_train)
y_pred = reg.predict(X_test_scaled)

In [105]:
print("score:",r2_score(y_pred,y_test))

score: 0.9796040748411479


In [106]:
## KNN yöntemi ile default olarak 5 gelen komşu sayısını 7 ile denedim. Fakat skore düştü.

In [107]:
## DecisionTree

In [108]:
from sklearn.tree import DecisionTreeRegressor

In [109]:
tree_model = DecisionTreeRegressor()

In [110]:
tree_model.fit(X_train_scaled,y_train)
y_pred = tree_model.predict(X_test_scaled)

In [111]:
print("score:",r2_score(y_pred,y_test))

score: 0.9999387199764357


In [112]:
# Tuning işlemine gerek kalmadan çok yüksek sonuç elde ettik.

## Özet

In [113]:
# Linear Regression: 0.49897085592909785
# PolynomialRegression : 0.9292154638502551  0.9969607071152742  0.9999637324440812  0.9999998840054525
# SVR : 0.9965754023715911
# KNN : 0.9850915688995806
# Decision Tree : 0.9999387199764357

In [114]:
# Bazen işimizi tek bir regresyon modeli çözemez. Farklı modelleri deneyerek ilerlememiz gerekir. Bu projede 5 farklı regresyon modelini kıyasladım.
# Tabiki her birinin ayrı ayrı özellikleri var. Amaç burada öğrendiklerimi pekiştirmekti.

In [ ]:
# İyi bir model için iyi bir EDA süreci ve Feature Engineering işlemi gereklidir. Bugünkü veri setinde temizlik ile zaman kaybetmeden sonuçları gördüm.